# Notebook 13 — Mixed-Regime Decomposition

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 12 exposed latency / throughput Pareto frontiers.

Notebook 13 asks a harder runtime question:

> what happens when a window is not one clean regime?

Instead of forcing each window into one label, this notebook decomposes windows into mixtures:

- coherent-local components
- sequential components
- SIMD-favorable random components
- zipfian heavy-tail components
- fragmented / clustered components

Constraint view:
> real streams often contain mixed structure; adaptive runtimes should estimate mixture, not only classify.

## Goals

1. Load Notebook 12 / Notebook 11 outputs when available.
2. Generate mixed-regime windows.
3. Compute structural feature vectors.
4. Build simple prototype vectors for each regime.
5. Estimate mixture weights using nonnegative least squares.
6. Compare hard classification vs mixture decomposition.
7. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import nnls

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Define regime prototypes

These prototypes summarize expected feature signatures from earlier notebooks.

In [ ]:
prototype_rows = [
    {
        "regime": "low_entropy_repeating",
        "entropy_norm": 0.10,
        "repetition_ratio": 0.98,
        "locality_small_delta_ratio": 1.00,
        "cache_window_reuse_proxy": 0.94,
        "branch_norm": 0.02,
        "coherence_score": 0.95,
        "hardware_pressure_proxy": 0.02,
    },
    {
        "regime": "sequential_ids",
        "entropy_norm": 0.90,
        "repetition_ratio": 0.00,
        "locality_small_delta_ratio": 1.00,
        "cache_window_reuse_proxy": 0.00,
        "branch_norm": 0.20,
        "coherence_score": 0.55,
        "hardware_pressure_proxy": 0.25,
    },
    {
        "regime": "uniform_32bit",
        "entropy_norm": 1.00,
        "repetition_ratio": 0.00,
        "locality_small_delta_ratio": 0.00,
        "cache_window_reuse_proxy": 0.00,
        "branch_norm": 0.72,
        "coherence_score": 0.08,
        "hardware_pressure_proxy": 0.88,
    },
    {
        "regime": "zipfian_smallints",
        "entropy_norm": 0.18,
        "repetition_ratio": 0.80,
        "locality_small_delta_ratio": 0.27,
        "cache_window_reuse_proxy": 0.35,
        "branch_norm": 0.72,
        "coherence_score": 0.42,
        "hardware_pressure_proxy": 0.72,
    },
    {
        "regime": "clustered_ranges",
        "entropy_norm": 0.55,
        "repetition_ratio": 0.98,
        "locality_small_delta_ratio": 0.00,
        "cache_window_reuse_proxy": 0.01,
        "branch_norm": 0.79,
        "coherence_score": 0.18,
        "hardware_pressure_proxy": 0.98,
    },
]

prototypes = pd.DataFrame(prototype_rows)
feature_cols = [
    "entropy_norm",
    "repetition_ratio",
    "locality_small_delta_ratio",
    "cache_window_reuse_proxy",
    "branch_norm",
    "coherence_score",
    "hardware_pressure_proxy",
]

prototypes

## Generate synthetic mixed windows

Each window has a true mixture vector. The observed feature vector is a noisy convex mixture of prototypes.

In [ ]:
rng = np.random.default_rng(42)
regimes = list(prototypes["regime"])

mixture_templates = [
    {"low_entropy_repeating": 0.80, "sequential_ids": 0.20},
    {"sequential_ids": 0.60, "uniform_32bit": 0.40},
    {"uniform_32bit": 0.70, "zipfian_smallints": 0.30},
    {"zipfian_smallints": 0.55, "clustered_ranges": 0.45},
    {"clustered_ranges": 0.70, "guarded_fallback_proxy": 0.0, "low_entropy_repeating": 0.30},
    {"low_entropy_repeating": 0.40, "zipfian_smallints": 0.40, "uniform_32bit": 0.20},
    {"sequential_ids": 0.35, "clustered_ranges": 0.35, "zipfian_smallints": 0.30},
    {"uniform_32bit": 0.50, "clustered_ranges": 0.50},
]

proto_matrix = prototypes.set_index("regime")[feature_cols]

rows = []
truth_weights = []

for i in range(160):
    template = mixture_templates[i % len(mixture_templates)]
    weights = {r: 0.0 for r in regimes}
    for k, v in template.items():
        if k in weights:
            weights[k] += v
    # Normalize.
    total = sum(weights.values())
    weights = {k: v / total for k, v in weights.items()}

    vec = sum(weights[r] * proto_matrix.loc[r].values for r in regimes)
    noisy = np.clip(vec + rng.normal(0, 0.025, size=len(feature_cols)), 0, 1)

    row = {"window_id": i}
    row.update({c: noisy[j] for j, c in enumerate(feature_cols)})
    row["dominant_truth_regime"] = max(weights, key=weights.get)
    rows.append(row)

    tw = {"window_id": i}
    tw.update({f"true_weight_{r}": weights[r] for r in regimes})
    truth_weights.append(tw)

mixed = pd.DataFrame(rows)
truth = pd.DataFrame(truth_weights)

mixed.head()

## Estimate mixture weights by nonnegative least squares

The model solves:

```text
observed_feature ≈ prototype_matrix × mixture_weights
```

with nonnegative weights normalized to sum to 1.

In [ ]:
A = prototypes[feature_cols].to_numpy().T  # features x regimes

estimates = []
reconstruction_rows = []

for _, row in mixed.iterrows():
    b = row[feature_cols].to_numpy(float)
    w, residual = nnls(A, b)
    if w.sum() > 0:
        w = w / w.sum()

    est = {"window_id": int(row["window_id"])}
    for regime, weight in zip(regimes, w):
        est[f"estimated_weight_{regime}"] = weight
    est["estimated_dominant_regime"] = regimes[int(np.argmax(w))]
    est["mixture_entropy"] = float(-(w[w > 0] * np.log2(w[w > 0])).sum())
    est["reconstruction_residual"] = float(residual)
    estimates.append(est)

    recon = A @ w
    rr = {"window_id": int(row["window_id"])}
    for c, val in zip(feature_cols, recon):
        rr[f"reconstructed_{c}"] = val
    reconstruction_rows.append(rr)

estimated = pd.DataFrame(estimates)
reconstructed = pd.DataFrame(reconstruction_rows)

result = mixed.merge(truth, on="window_id").merge(estimated, on="window_id").merge(reconstructed, on="window_id")
result.head()

## Mixture-aware policy recommendation

Instead of one hard regime, choose policy from estimated mixture:

- coherent-local if low-entropy repeating dominates
- SIMD if uniform dominates
- guarded fallback if clustered dominates
- hybrid when mixture entropy is high
- scalar/hybrid for sequential-heavy mixtures

In [ ]:
def recommend_policy(row):
    weights = {r: row[f"estimated_weight_{r}"] for r in regimes}
    dominant = max(weights, key=weights.get)
    entropy = row["mixture_entropy"]

    if entropy > 1.45:
        return "hybrid"
    if dominant == "low_entropy_repeating":
        return "coherent_local"
    if dominant == "uniform_32bit":
        return "simd"
    if dominant == "clustered_ranges":
        return "guarded_fallback"
    if dominant == "sequential_ids":
        return "hybrid"
    if dominant == "zipfian_smallints":
        return "hybrid"
    return "hybrid"

result["mixture_policy"] = result.apply(recommend_policy, axis=1)

hard_policy_map = {
    "low_entropy_repeating": "coherent_local",
    "sequential_ids": "hybrid",
    "uniform_32bit": "simd",
    "zipfian_smallints": "hybrid",
    "clustered_ranges": "guarded_fallback",
}
result["hard_policy_from_dominant"] = result["estimated_dominant_regime"].map(hard_policy_map).fillna("hybrid")
result["policy_changed_by_mixture_entropy"] = result["mixture_policy"] != result["hard_policy_from_dominant"]

result[["window_id", "dominant_truth_regime", "estimated_dominant_regime", "mixture_entropy", "mixture_policy"]].head()

## Export mixed-regime decomposition table

In [ ]:
csv_path = RESULTS_DIR / "notebook13_mixed_regime_decomposition.csv"
json_path = RESULTS_DIR / "notebook13_mixed_regime_decomposition.json"

result.to_csv(csv_path, index=False)
result.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Estimated mixture weights over time

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook13_estimated_mixture_weights.png"

weight_cols = [f"estimated_weight_{r}" for r in regimes]

plt.figure(figsize=(12, 5))
bottom = np.zeros(len(result))
x = result["window_id"].values
for regime, col in zip(regimes, weight_cols):
    plt.bar(x, result[col].values, bottom=bottom, label=regime, width=1.0)
    bottom += result[col].values

plt.xlabel("Window")
plt.ylabel("Estimated mixture weight")
plt.title("Mixed-Regime Decomposition: Estimated Weights")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — True vs estimated dominant regime

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook13_dominant_regime_timeline.png"

labels = sorted(set(result["dominant_truth_regime"]).union(set(result["estimated_dominant_regime"])))
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 5))
plt.step(result["window_id"], result["dominant_truth_regime"].map(lab_to_id), where="mid", label="true dominant")
plt.step(result["window_id"], result["estimated_dominant_regime"].map(lab_to_id), where="mid", label="estimated dominant", linestyle="--")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Dominant regime")
plt.title("Mixed-Regime Decomposition: True vs Estimated Dominant Regime")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Mixture entropy timeline

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook13_mixture_entropy_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(result["window_id"], result["mixture_entropy"])
plt.xlabel("Window")
plt.ylabel("Mixture entropy")
plt.title("Mixed-Regime Decomposition: Mixture Entropy Over Time")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Reconstruction residuals

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook13_reconstruction_residuals.png"

plt.figure(figsize=(12, 4))
plt.plot(result["window_id"], result["reconstruction_residual"])
plt.xlabel("Window")
plt.ylabel("NNLS reconstruction residual")
plt.title("Mixed-Regime Decomposition: Reconstruction Residuals")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Policy from mixture decomposition

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook13_mixture_policy_timeline.png"

policies = sorted(result["mixture_policy"].unique())
pol_to_id = {p: i for i, p in enumerate(policies)}

plt.figure(figsize=(12, 4))
plt.step(result["window_id"], result["mixture_policy"].map(pol_to_id), where="mid")
plt.yticks(list(pol_to_id.values()), list(pol_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Mixture-aware policy")
plt.title("Mixed-Regime Decomposition: Policy Timeline")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Average estimated mixture by true dominant regime

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook13_average_mixture_by_regime.png"

avg = result.groupby("dominant_truth_regime")[weight_cols].mean()

plt.figure(figsize=(10, 5))
plt.imshow(avg.values, aspect="auto", vmin=0, vmax=1)
plt.yticks(range(len(avg.index)), avg.index)
plt.xticks(range(len(weight_cols)), [c.replace("estimated_weight_", "") for c in weight_cols], rotation=45, ha="right")
plt.colorbar(label="Average estimated weight")
plt.title("Average Estimated Mixture by True Dominant Regime")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_13_mixed_regime_decomposition.md"

dominant_accuracy = float((result["dominant_truth_regime"] == result["estimated_dominant_regime"]).mean())

summary = {
    "windows": int(len(result)),
    "dominant_regime_accuracy": dominant_accuracy,
    "mean_mixture_entropy": float(result["mixture_entropy"].mean()),
    "mean_reconstruction_residual": float(result["reconstruction_residual"].mean()),
    "policy_changed_by_mixture_entropy_rate": float(result["policy_changed_by_mixture_entropy"].mean()),
}

policy_counts = result["mixture_policy"].value_counts().rename_axis("policy").reset_index(name="count")

lines = [
    "# Report 13 — Mixed-Regime Decomposition",
    "",
    "This report estimates mixed-regime structure within streaming windows instead of forcing each window into one hard label.",
    "",
    "Constraint view:",
    "> real streams often contain mixed structure; adaptive runtimes should estimate mixture, not only classify.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Mixture-aware policy counts",
    "",
    policy_counts.to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Mixture weights expose blended windows that hard classification hides.",
    "- Mixture entropy identifies when runtime policy should hesitate or choose hybrid paths.",
    "- Reconstruction residuals identify windows poorly explained by current prototypes.",
    "- This notebook turns regime classification into regime decomposition.",
    "",
    "## Next step",
    "",
    "Notebook 14 can introduce online drift detection: detect when new or changing mixtures stop matching existing prototypes.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook13_mixed_regime_decomposition_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook13_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_13_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))